# Proyecto 2 (alternativa sin instalar nada): SQL con DuckDB

Las mismas consultas de `analisis_sql.sql`, pero corriendo con **DuckDB**: un motor SQL que funciona dentro de Python, sin servidor ni base de datos que instalar. Sirve en Google Colab y en tu compu.

Los resultados de *ventas por mes* tienen que coincidir con los del notebook de Pandas.

In [1]:
%pip install -q duckdb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: C:\Users\m_cas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import duckdb

# Si ventas.csv está en la misma carpeta (corriendo local) lo usamos;
# si no (por ejemplo en Google Colab), lo bajamos directo de GitHub.
URL_CSV = "https://raw.githubusercontent.com/Academia-Mariana-Casella/masterclass-data-analytics/main/ventas.csv"
ruta = "ventas.csv" if os.path.exists("ventas.csv") else URL_CSV

# Creamos la tabla ventas. La fecha viene como día/mes/año, así que se lo indicamos.
duckdb.sql(f"""
    CREATE OR REPLACE TABLE ventas AS
    SELECT * FROM read_csv('{ruta}', header = true, dateformat = '%d/%m/%Y',
                           columns = {{'id_venta': 'TEXT', 'fecha': 'DATE', 'canal': 'TEXT',
                                      'categoria': 'TEXT', 'monto': 'DOUBLE', 'vendedor': 'TEXT'}})
""")

CountBIGINT
1850


## 1. Entender el archivo

In [3]:
duckdb.sql("SELECT * FROM ventas LIMIT 8").df()

,id_venta,fecha,canal,categoria,monto,vendedor
0,V01427,2025-09-02,Local,Calzado,10100.0,Lucía R.
1,V00209,2025-09-02,Local,Calzado,4500.0,Sofía G.
2,V00822,2025-09-02,Online,Calzado,16100.0,Sofía G.
3,V00073,2025-09-03,Local,Indumentaria,18900.0,Martín P.
4,V01516,2025-09-03,Local,Calzado,11800.0,Martín P.
5,V00129,2025-09-03,Mayorista,Accesorios,19500.0,Sofía G.
6,V01169,2025-09-03,Online,Calzado,NaN,Martín P.
7,V01484,2025-09-03,Online,Hogar,4300.0,Martín P.


In [4]:
duckdb.sql("""
    SELECT COUNT(*) AS filas, MIN(fecha) AS fecha_desde, MAX(fecha) AS fecha_hasta
    FROM ventas
""").df()

,filas,fecha_desde,fecha_hasta
0,1850,2025-09-02,2026-10-21


## 2. Revisar los montos faltantes

No los completamos con 0: `SUM(monto)` ya ignora los NULL.

In [5]:
duckdb.sql("""
    SELECT
        COUNT(*) FILTER (WHERE monto IS NULL) AS ventas_sin_monto,
        COUNT(*) AS total_ventas,
        ROUND(100.0 * COUNT(*) FILTER (WHERE monto IS NULL) / COUNT(*), 1) AS porcentaje_faltante
    FROM ventas
""").df()

,ventas_sin_monto,total_ventas,porcentaje_faltante
0,74,1850,4.0


## 3. Ventas por mes

In [6]:
duckdb.sql("""
    SELECT
        strftime(fecha, '%Y-%m') AS mes,
        SUM(monto) AS monto_total,
        COUNT(*) AS cantidad_ventas,
        COUNT(*) FILTER (WHERE monto IS NULL) AS ventas_sin_monto_en_el_mes
    FROM ventas
    GROUP BY mes
    ORDER BY mes
""").df()

,mes,monto_total,cantidad_ventas,ventas_sin_monto_en_el_mes
0,2025-09,1806500.0,124,5
1,2025-10,2219100.0,152,8
2,2025-11,1531700.0,120,6
3,2025-12,3636100.0,277,7
4,2026-01,786000.0,58,4
5,2026-02,1424900.0,106,4
6,2026-03,1710800.0,116,3
7,2026-04,1631200.0,117,3
8,2026-05,1873100.0,142,6
9,2026-06,1593600.0,123,6


## 4. Mes de mayor venta

In [7]:
duckdb.sql("""
    SELECT strftime(fecha, '%Y-%m') AS mes, SUM(monto) AS monto_total
    FROM ventas
    GROUP BY mes
    ORDER BY monto_total DESC
    LIMIT 1
""").df()

,mes,monto_total
0,2025-12,3636100.0


## 5. Chequeo de períodos completos

Para no comparar un mes completo contra uno que recién empezó.

In [8]:
duckdb.sql("""
    SELECT strftime(fecha, '%Y-%m') AS mes, MAX(day(fecha)) AS ultimo_dia_con_datos
    FROM ventas
    GROUP BY mes
    ORDER BY mes
""").df()

,mes,ultimo_dia_con_datos
0,2025-09,30
1,2025-10,31
2,2025-11,29
3,2025-12,31
4,2026-01,31
5,2026-02,28
6,2026-03,31
7,2026-04,30
8,2026-05,31
9,2026-06,30
